# CMSC · AI ACOUSTIC — Fine-Tuning LoRA en Google Colab (GPU T4 o Fallback Seguro en CPU)

* **Proyecto:** `sigcTiArural` · Módulo `CMSC · AI ACOUSTIC`
* **Propietario:** Bernardo Adolfo Gómez Montoya (`ServSoftBagm` / `IRTELC01`)
* **Repositorio Canónico Drive:** `/content/drive/MyDrive/Gobernanza_Portafolio_Badolgm_IRTELC01_2026/03_Continuidad_sigcTiArural_y_Ecosistema/CMSC_AI_ACOUSTIC_REPO_V1`

> **Nota:** Este notebook detecta automáticamente si hay **GPU CUDA** (usa **QLoRA 4-bit NF4 + fp16**) o si estás en **CPU** (evita `bitsandbytes` 4-bit y usa **LoRA estándar en float32** o valida el pipeline sin romperse por falta de GPU).

In [ ]:
# QUÉ: Monta Google Drive y verifica las carpetas canónicas del repositorio.
# POR QUÉ: Persiste checkpoints y manifiestos directamente en Drive.
# INVARIANTE: Modo estrictamente solo lectura sobre 02_DATASETS_Y_CONTRATOS_V1/.

import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

REPO_ROOT = Path("/content/drive/MyDrive/Gobernanza_Portafolio_Badolgm_IRTELC01_2026/03_Continuidad_sigcTiArural_y_Ecosistema/CMSC_AI_ACOUSTIC_REPO_V1")
DIR_DOCS = REPO_ROOT / "01_DOCS_CANONICOS_Y_ADR"
DIR_DATASETS = REPO_ROOT / "02_DATASETS_Y_CONTRATOS_V1"
DIR_NOTEBOOKS = REPO_ROOT / "03_NOTEBOOKS_COLAB_ENTRENAMIENTO"
DIR_MODELS = REPO_ROOT / "04_MODELOS_LLMS_Y_ADAPTADORES"
DIR_LOGS = REPO_ROOT / "05_EVALUACION_BENCHMARKS_Y_LOGS"

for d in [DIR_DOCS, DIR_DATASETS, DIR_NOTEBOOKS, DIR_MODELS, DIR_LOGS]:
    assert d.exists(), f"No se encontró la carpeta esperada en Drive: {d}"

print("Repositorio canónico verificado:", REPO_ROOT)

Mounted at /content/drive
Repositorio canónico verificado: /content/drive/MyDrive/Gobernanza_Portafolio_Badolgm_IRTELC01_2026/03_Continuidad_sigcTiArural_y_Ecosistema/CMSC_AI_ACOUSTIC_REPO_V1


In [ ]:
# QUÉ: Instala dependencias compatibles con GPU o CPU según el acelerador activo.
# POR QUÉ: Evita fallos de bitsandbytes cuando Colab asigna entorno CPU por cuota de GPU.
# INVARIANTE: No modifica archivos en Drive.

import torch

HAS_CUDA = torch.cuda.is_available()
print("Acelerador detectado:", f"GPU ({torch.cuda.get_device_name(0)})" if HAS_CUDA else "CPU (sin CUDA)")

if HAS_CUDA:
    !pip install -q -U transformers peft trl bitsandbytes accelerate datasets
else:
    !pip install -q -U transformers peft trl accelerate datasets

Acelerador detectado: CPU (sin CUDA)


In [ ]:
# QUÉ: Verifica SHA-256 e instantánea de 02_DATASETS_Y_CONTRATOS_V1/ antes de entrenar.
# POR QUÉ: Garantiza integridad del dataset y detecta cualquier alteración accidental.
# INVARIANTE: Ningún archivo en 02_DATASETS_Y_CONTRATOS_V1/ puede ser modificado.

import hashlib
import json

DATASET_PATH = DIR_DATASETS / "dataset_acoustic_sft_v1.jsonl"
SHA_PATH = DIR_DATASETS / "dataset_acoustic_sft_v1.sha256"

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def snapshot_dir(directory: Path) -> dict:
    return {p.name: sha256_file(p) for p in sorted(directory.iterdir()) if p.is_file()}

assert DATASET_PATH.exists(), f"Falta {DATASET_PATH} en 02_DATASETS_Y_CONTRATOS_V1/"

dataset_sha256 = sha256_file(DATASET_PATH)
if SHA_PATH.exists():
    expected_sha = SHA_PATH.read_text(encoding="utf-8").strip().split()[0]
    assert dataset_sha256 == expected_sha, f"Hash SHA-256 no coincide: {dataset_sha256} != {expected_sha}"

snapshot_before = snapshot_dir(DIR_DATASETS)
records = [json.loads(line) for line in DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"Dataset validado: {len(records)} registros | SHA-256: {dataset_sha256}")

Dataset validado: 17 registros | SHA-256: 96b330c4ead33f257cfd73282b41f38a5dd5b13600d49a6b834b024d3e94485e


In [ ]:
# QUÉ: Entrena adaptador LoRA adaptándose automáticamente a GPU (QLoRA 4-bit) o CPU (LoRA float32).
# POR QUÉ: Resuelve tanto el bloqueo por assert de CUDA como el error de bitsandbytes/fp16 en CPU.
# INVARIANTE: Guarda checkpoints en 04_MODELOS_LLMS_Y_ADAPTADORES/ sin tocar 02_DATASETS_Y_CONTRATOS_V1/.

from datetime import datetime, timezone
import inspect
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

HAS_CUDA = torch.cuda.is_available()
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct" if HAS_CUDA else "Qwen/Qwen2.5-0.5B-Instruct"
RUN_NAME = "cmsc_acoustic_qwen25_qlora_v1" if HAS_CUDA else "cmsc_acoustic_qwen25_05b_cpu_lora_v1"
OUTPUT_DIR = DIR_MODELS / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if HAS_CUDA:
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=False,
    )
else:
    print("AVISO: Ejecutando en CPU (sin cuota GPU). Se usa Qwen2.5-0.5B-Instruct + LoRA en float32.")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
        trust_remote_code=False,
    )

def format_example(row: dict) -> dict:
    prompt = row.get("instruction") or row.get("prompt") or ""
    inp = row.get("input") or ""
    resp = row.get("output") or row.get("response") or ""
    user_msg = f"{prompt}\n\nContexto:\n{inp}".strip() if inp else prompt
    messages = [
        {"role": "system", "content": "Eres el analista acústico de CMSC sigcTiArural. Respeta estrictamente C0, C1 (REF) y rules-v0 (SIM). Nunca fabriques métricas."},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": resp},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

raw_ds = Dataset.from_list([format_example(r) for r in records])
split_ds = raw_ds.train_test_split(test_size=0.15, seed=42) if len(raw_ds) >= 6 else {"train": raw_ds, "test": raw_ds}

peft_config = LoraConfig(
    r=16 if HAS_CUDA else 8,
    lora_alpha=32 if HAS_CUDA else 16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=4 if HAS_CUDA else 2,
    per_device_train_batch_size=2 if HAS_CUDA else 1,
    gradient_accumulation_steps=4 if HAS_CUDA else 4,
    learning_rate=2e-4,
    fp16=bool(HAS_CUDA),
    use_cpu=not HAS_CUDA,
    logging_steps=2,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    eval_strategy="epoch",
    report_to="none",
    seed=42,
)

sft_params = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = dict(
    model=model,
    train_dataset=split_ds["train"],
    eval_dataset=split_ds["test"],
    peft_config=peft_config,
    args=training_args,
)
if "processing_class" in sft_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer
if "dataset_text_field" in sft_params:
    trainer_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in sft_params:
    trainer_kwargs["max_seq_length"] = 1024 if HAS_CUDA else 512

trainer = SFTTrainer(**trainer_kwargs)

checkpoints = sorted(OUTPUT_DIR.glob("checkpoint-*"))
resume_ckpt = str(checkpoints[-1]) if checkpoints else None
train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
eval_metrics = trainer.evaluate()

FINAL_ADAPTER_DIR = OUTPUT_DIR / "final_adapter"
trainer.model.save_pretrained(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))
print("Adaptador final guardado en:", FINAL_ADAPTER_DIR)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (15 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,2.992279,2.886970,2.659663,0.494845,6017.000000
2,2.598626,2.662067,2.598879,0.519429,12034.000000


Training Loss,Validation Loss,Epoch,Entropy,Mean Token Accuracy,Num Tokens
2.598626,2.662067,2,2.598879,0.519429,12034.000000


Adaptador final guardado en: /content/drive/MyDrive/Gobernanza_Portafolio_Badolgm_IRTELC01_2026/03_Continuidad_sigcTiArural_y_Ecosistema/CMSC_AI_ACOUSTIC_REPO_V1/04_MODELOS_LLMS_Y_ADAPTADORES/cmsc_acoustic_qwen25_05b_cpu_lora_v1/final_adapter


In [ ]:
# QUÉ: Audita integridad post-entrenamiento de 02_DATASETS_Y_CONTRATOS_V1/ y escribe manifest.json + reporte.
# POR QUÉ: Mantiene trazabilidad completa para el cargador local (Prompt 3).
# INVARIANTE: Ningún archivo en 02_DATASETS_Y_CONTRATOS_V1/ pudo alterarse.

snapshot_after = snapshot_dir(DIR_DATASETS)
assert snapshot_before == snapshot_after, "ALERTA: Cambió un archivo en 02_DATASETS_Y_CONTRATOS_V1/"

adapter_files = sorted([p.name for p in FINAL_ADAPTER_DIR.iterdir() if p.is_file()])
manifest = {
    "run_name": RUN_NAME,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "accelerator": "cuda" if HAS_CUDA else "cpu",
    "base_model_id": BASE_MODEL_ID,
    "dataset_file": DATASET_PATH.name,
    "dataset_sha256": dataset_sha256,
    "dataset_records": len(records),
    "adapter_dir": str(FINAL_ADAPTER_DIR),
    "required_files": adapter_files,
    "metrics": {
        "train_loss": float(train_result.training_loss),
        "eval_loss": float(eval_metrics.get("eval_loss", -1.0)),
    },
    "governance": {
        "evaluation_layer": "C0",
        "reference_layer": "C1",
        "deterministic_method": "rules-v0",
    },
}

(FINAL_ADAPTER_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
(DIR_LOGS / f"{RUN_NAME}_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

report_md = f"""# Reporte de Entrenamiento Colab — {RUN_NAME}
- **Fecha UTC:** {manifest['timestamp_utc']}
- **Acelerador:** `{manifest['accelerator']}`
- **Modelo base:** `{BASE_MODEL_ID}`
- **Dataset SHA-256:** `{dataset_sha256}` ({len(records)} pares)
- **Train Loss:** `{manifest['metrics']['train_loss']:.4f}`
- **Eval Loss:** `{manifest['metrics']['eval_loss']:.4f}`
- **Integridad `02_DATASETS_Y_CONTRATOS_V1/`:** VERIFICADA SIN CAMBIOS
"""
(DIR_LOGS / f"{RUN_NAME}_training_report.md").write_text(report_md, encoding="utf-8")
print("Manifiesto y reporte guardados correctamente.")

Manifiesto y reporte guardados correctamente.
